# VTA figure

This notebook contains the final plotting code retained from the actual analysis notebook. It expects the CSV outputs generated by `03_run_vta_robustness_analyses.py`.


In [ ]:
# ============================================================
# VTA rebuttal figure: clean LISBET / Nature-style version
# Updated:
# - panel a = bar plot
# - panel labels a, b, c = lowercase and not bold
# - compact spacing
# - all statistics boxes in the upper right
# - panel b inset moved upward but kept below the stats box
# - forced white background
# ============================================================

from pathlib import Path
import tarfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 0. Find or extract results folder
# ============================================================

HOME = Path.home()

TARGET_PARENT = HOME / "Dokumente" / "Lisbet"
TARGET_PARENT.mkdir(parents=True, exist_ok=True)

search_places = [
    Path.cwd(),
    HOME / "Downloads",
    HOME / "Dokumente",
    HOME / "Dokumente" / "Lisbet",
    HOME / "Documents",
    HOME / "Documents" / "Lisbet",
]

folder_hits = []
tar_hits = []

for place in search_places:
    if not place.exists():
        continue
    folder_hits += [p for p in place.rglob("robust_residual_analysis_final") if p.is_dir()]
    tar_hits += [p for p in place.rglob("robust_residual_analysis_final.tar.xz") if p.is_file()]

ROOT = None

for p in folder_hits:
    if list(p.glob("*.csv")):
        ROOT = p
        break

if ROOT is None and tar_hits:
    tar_path = tar_hits[0]
    print("Extracting:", tar_path)
    with tarfile.open(tar_path, "r:xz") as tar:
        tar.extractall(TARGET_PARENT)
    ROOT = TARGET_PARENT / "robust_residual_analysis_final"

if ROOT is None:
    raise FileNotFoundError(
        "Could not find robust_residual_analysis_final or robust_residual_analysis_final.tar.xz. "
        "Move the .tar.xz file to ~/Dokumente/Lisbet/ and rerun."
    )

print("Using ROOT:", ROOT)

print("\nCSV files found:")
for p in sorted(ROOT.glob("*.csv")):
    print(" -", p.name)

OUTDIR = ROOT / "figures_lisbet_style_clean"
OUTDIR.mkdir(parents=True, exist_ok=True)

DPI = 1000

# ============================================================
# 1. Load LISBET style but force clean white background
# ============================================================

style_candidates = [
    Path("lisbet.mplstyle"),
    HOME / "Dokumente" / "Lisbet" / "lisbet.mplstyle",
    ROOT / "lisbet.mplstyle",
]

for s in style_candidates:
    if s.exists():
        plt.style.use(str(s))
        print("\nLoaded style:", s)
        break

# Force white/light Nature-like style even if LISBET style is dark
mpl.rcParams.update({
    "figure.facecolor": "white",
    "figure.edgecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "savefig.transparent": False,
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "grid.color": "0.85",
    "axes.grid": False,
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def clean_axis(ax):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax

def panel_label(ax, label):
    ax.text(
        -0.16, 1.05,
        label.lower(),
        transform=ax.transAxes,
        fontsize=11,
        fontweight="normal",
        ha="left",
        va="top",
        color="black"
    )

def stats_box(ax, text):
    ax.text(
        0.98, 0.98,
        text,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=7,
        color="black",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="0.8",
            linewidth=0.6
        )
    )

# ============================================================
# 2. Load data
# ============================================================

required_files = {
    "model": "main_and_extended_model_comparison_fast.csv",
    "residual_model": "residual_model_summary_fast.csv",
    "boot": "trial_cluster_bootstrap_model_comparison_fast.csv",
    "loo": "leave_one_trial_out_model_comparison_fast.csv",
    "proto": "residual_by_prototype_trial_bootstrap_ci_fast.csv",
    "perm": "within_trial_permutation_residual_delta_r2_fast.csv",
}

for key, fname in required_files.items():
    if not (ROOT / fname).exists():
        raise FileNotFoundError(f"Missing required file: {ROOT / fname}")

model = pd.read_csv(ROOT / required_files["model"])
residual_model = pd.read_csv(ROOT / required_files["residual_model"])
boot = pd.read_csv(ROOT / required_files["boot"])
loo = pd.read_csv(ROOT / required_files["loo"])
proto = pd.read_csv(ROOT / required_files["proto"])
perm = pd.read_csv(ROOT / required_files["perm"])

main = model[model["analysis"] == "main_velocity_distance_vs_lisbet"].iloc[0]
resid_main = residual_model[residual_model["analysis"] == "residual_after_velocity_distance"].iloc[0]

# Main model values
r2_m1 = float(main["r2_m1"])
r2_m2 = float(main["r2_m2"])
delta_r2 = float(main["delta_r2"])
f_stat = float(main["f_stat"])
f_df_num = int(main["f_df_num"])
f_df_den = int(main["f_df_den"])
f_p = float(main["f_p"])
delta_aic = float(main["delta_aic_m1_minus_m2"])

# Residual model values
delta_resid_r2 = float(resid_main["delta_residual_r2"])
resid_p = float(resid_main["f_p"])
perm_p = float(perm["permutation_p_value"].iloc[0])

# Convert to percentage points
observed_delta_pp = delta_r2 * 100
boot_delta_pp = boot["delta_r2"].astype(float) * 100
loo_delta_pp = loo["delta_r2"].astype(float) * 100

# Prototype residuals
proto = proto.sort_values("prototype_id").copy()
proto["prototype_id"] = proto["prototype_id"].astype(int)

# ============================================================
# 3. Build figure
# ============================================================

fig = plt.figure(figsize=(10.6, 3.8), facecolor="white")
fig.patch.set_facecolor("white")

gs = fig.add_gridspec(
    1, 3,
    width_ratios=[1.0, 1.1, 1.15],
    wspace=0.28
)

# ============================================================
# a. Model comparison: bar plot
# ============================================================

axA = fig.add_subplot(gs[0, 0])
axA.set_facecolor("white")
panel_label(axA, "a")

x = np.array([0, 1])
vals = np.array([r2_m1, r2_m2]) * 100

labels = [
    "Velocity\n+ distance",
    "Velocity + distance\n+ LISBET prototype"
]

axA.bar(x, vals, width=0.58)

for xi, yi in zip(x, vals):
    axA.text(
        xi,
        yi + 0.01,
        f"{yi:.3f}%",
        ha="center",
        va="bottom",
        fontsize=7,
        color="black"
    )

axA.set_xticks(x)
axA.set_xticklabels(labels)
axA.set_ylabel(r"Explained variance, $R^2$ (%)")
axA.set_title("LISBET adds information")
clean_axis(axA)

axA.set_ylim(min(vals) - 0.05, max(vals) + 0.12)

stats_box(
    axA,
    (
        f"ΔR² = {observed_delta_pp:.3f} percentage points\n"
        f"F({f_df_num}, {f_df_den}) = {f_stat:.2f}\n"
        f"p = {f_p:.2e}\n"
        f"ΔAIC = {delta_aic:.2f}"
    )
)

# ============================================================
# b. Bootstrap robustness + leave-one-trial-out inset
# ============================================================

axB = fig.add_subplot(gs[0, 1])
axB.set_facecolor("white")
panel_label(axB, "b")

axB.hist(
    boot_delta_pp,
    bins=28,
    edgecolor="black",
    linewidth=0.35
)

axB.axvline(observed_delta_pp, linewidth=1.2, linestyle="-")
axB.axvline(np.quantile(boot_delta_pp, 0.025), linestyle="--", linewidth=1)
axB.axvline(np.quantile(boot_delta_pp, 0.975), linestyle="--", linewidth=1)
axB.axvline(0, linestyle=":", linewidth=1)

q025, q50, q975 = np.quantile(boot_delta_pp, [0.025, 0.5, 0.975])
positive_fraction = (boot_delta_pp > 0).mean() * 100

axB.set_xlabel(r"Bootstrapped $\Delta R^2$ (%)")
axB.set_ylabel("Count")
axB.set_title("Robustness across trials")
clean_axis(axB)

stats_box(
    axB,
    (
        f"median = {q50:.3f}%\n"
        f"95% CI = [{q025:.3f}, {q975:.3f}]%\n"
        f"positive = {positive_fraction:.1f}%"
    )
)

# inset moved higher but below the upper-right stats box
axins = axB.inset_axes([0.57, 0.28, 0.35, 0.25])
axins.set_facecolor("white")

x_loo = np.arange(1, len(loo_delta_pp) + 1)

axins.plot(
    x_loo,
    loo_delta_pp,
    marker="o",
    markersize=1.6,
    linewidth=0.8
)

axins.axhline(0, linestyle="--", linewidth=0.8)
axins.axhline(observed_delta_pp, linestyle=":", linewidth=0.8)

axins.set_title("Leave-one-trial-out", fontsize=6.3, pad=2, color="black")
axins.tick_params(labelsize=5.5, length=2, colors="black")
axins.spines["top"].set_visible(False)
axins.spines["right"].set_visible(False)
axins.spines["left"].set_color("black")
axins.spines["bottom"].set_color("black")

# ============================================================
# c. Residual VTA signal by prototype
# ============================================================

axC = fig.add_subplot(gs[0, 2])
axC.set_facecolor("white")
panel_label(axC, "c")

x = proto["prototype_id"].values
y = proto["mean_residual"].values
yerr_low = y - proto["ci95_low"].values
yerr_high = proto["ci95_high"].values - y

axC.errorbar(
    x,
    y,
    yerr=np.vstack([yerr_low, yerr_high]),
    fmt="o-",
    linewidth=1.0,
    markersize=4.0,
    capsize=2.5
)

axC.axhline(0, linestyle="--", linewidth=1)

axC.set_xlabel("Prototype ID")
axC.set_ylabel("Residual VTA activity")
axC.set_title("Prototype-specific residual signal")
clean_axis(axC)

if "n" in proto.columns:
    n_threshold = np.quantile(proto["n"].values, 0.75)
    for xi, yi, ni in zip(x, y, proto["n"].values):
        if ni >= n_threshold:
            axC.text(
                xi,
                yi,
                f" n={int(ni)}",
                fontsize=6,
                va="bottom",
                ha="left",
                color="black"
            )

stats_box(
    axC,
    (
        f"Residual model:\n"
        f"ΔR² = {delta_resid_r2*100:.3f}%\n"
        f"p = {resid_p:.2e}\n"
        f"Permutation p = {perm_p:.4f}"
    )
)

# ============================================================
# 4. Save
# ============================================================

png_path = OUTDIR / "vta_rebuttal_abc_clean_barplot_compact_upperright.png"
pdf_path = OUTDIR / "vta_rebuttal_abc_clean_barplot_compact_upperright.pdf"

fig.savefig(
    png_path,
    dpi=DPI,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

fig.savefig(
    pdf_path,
    dpi=DPI,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

plt.show()

print("\nSaved:")
print(png_path)
print(pdf_path)